## DSAN 6000 Homework 4A: I/O Speed and Memory Efficiency with DuckDB

## Overview

In the very first week of class, when you first saw the following diagram:

<center>

<img src='images/big-data-definition.svg' width='60%'></img>

</center>

You may have had your Pandas hat on, and you may have thought (as I did when I first learned this material!) something like:

> *Wait... if I want to download and analyze a data file from some website, but I hit the wall that the data file is too big to be **stored** on a single computer... am I not just out of luck?*
> 
> *Because, in the abstract, I'd love to "chop" this data file into smaller pieces. But in practice, wouldn't I need to **download it** onto a computer in the first place, so that this computer could then run some **code** or **process** to actually **do** the "chopping"?*

After some deeper-diving and/or asking AI things about that question, you would hopefully resolve these fears a bit, by learning how:

* (a) Functions like `pd.read_csv()` *do* have arguments like `chunksize` that could allow reading a file in pieces, and
* (b) These functions also support reading from **remote** sources like URLs or S3 bucket URIs.

Nevertheless, you would still be faced with the somewhat-daunting problem of figuring out what value to use for the `chunksize` argument, since different data values generally require different amounts of storage space: if you were downloading a `.csv` file where each row contained the text of a historical novel, for example, your choice of `chunksize` might force Pandas to stop reading right in the middle of the single row containing the 1.2 million words of Proust's [*À la Recherche du temps perdu*](https://observablehq.com/@guillaume-lesaine/in-search-of-marcel-proust) 😱

The point of this introduction is just to scare you away from "manually" optimizing chunk-by-chunk reading of `.csv` files, and towards embracing the combination of the `.parquet` format and DuckDB, given one of DuckDB's most powerful features: the ability to

* **Read file metadata** before loading the actual contents of the file, and then
* **Use** this metadata to **optimize the loading and processing of the file's contents.**

With this combination, you can "hand" a query over to DuckDB (note the full separation of computation from data that this implies!), and it will figure out how to optimize RAM usage on your computer, by loading **only the specific *subset* of the full data file content** that is necessary to satisfy the query.

In Part 1 you will see what this looks like concretely, for the ACLED event data mentioned in the `README.md` overview.

## Part 1: Querying File *Metadata* From S3

As mentioned in class during Week 4, one of the biggest drawbacks of the `.csv` format from a *Data Engineering* perspective is its lack of **embedded metadata** that could be used to plan out the optimal execution of a query *in advance*.

Given just a `.csv` file, a Data Engineer has no way of knowing (for example) whether they can immediately start computing means and averages of certain columns as rows are being read into memory line-by-line, or whether some conversion (say, parsing of timestamps) or missing-data handling will have to be carried out first, *after* the entire file has been read into memory.

As was *also* mentioned in Week 4, however, data files in the `.parquet` format always include **built-in** schema information, in the form of **metadata** stored at the end of the file.

> **The Parquet Metadata *Footer***
> 
> If you're curious, there's a technical reason for storing metadata at the *end* rather than the beginning of the `.parquet` file: storage setups like S3 only support **sequential writes** and **don't** support "editing" or "updating" the contents of a file after its bytes have been written (all files in S3 are immutable). Thus, efficient creation of a `.parquet` file from some prior data format involves (1) writing the data values into an S3 object as bytes, sequentially, computing statistics about these bytes along the way, and then (2) writing the final statistics – like the number of missing values – at the very end.
> 
> In other words, given how S3 objects work, there is no way to "go back up" to the top of the file to write this metadata, since data objects stored in S3 in fact have no "edit" or "update" mode at all. If you use `boto3` or some other API to "add" bytes to the beginning of an S3 object, what is really happening under the hood is that an **entirely new file** is being created: the bytes you're hoping to add to the "beginning" of the original file are then written first, followed by the remaining bytes from the original file, and the resulting *new* file is given the same name as the previous file (which is then discarded, unless you're paying extra for S3's version control features!)

## Part 1: Querying Files Directly from S3

In [ ]:
#| label: Q1-response


## Part 2: Optimizing Your Pandas Workflow

In [ ]:
#| label: Q2-response
